# **HospitalityGPT - Retrieval-Augmented Generation (RAG) Chatbot**

## **Project Objective**

The objective of this notebook is to build an intelligent HospitalityGPT chatbot by combining the fine-tuned TinyLlama model with Retrieval-Augmented Generation (RAG).

Instead of relying solely on the model's learned knowledge, the chatbot first retrieves the most relevant hospitality information from a knowledge base before generating a response.

This approach improves factual accuracy, reduces hallucinations, and enables the chatbot to provide context-aware hospitality support.

The RAG pipeline works in five steps.

1. The user asks a question.
2. The question is converted into a semantic embedding using Sentence Transformers.
3. That embedding is compared against a FAISS vector database. Why? Because FAISS (Facebook AI Similarity Search) efficiently searches through thousands of vector embeddings to find the documents most similar to the user's query which makes retrieval extremely fast.
4. The most relevant hospitality documents are retrieved.
5. Those documents are added to the prompt before the language model generates its answer.

This provides additional context and helps produce more accurate responses.

## **Github Link**

https://github.com/AimanSahay/HospitalityGPT-Industry-Specific-Hospitality-Support-Assistant

**Install Required Libraries**

This notebook uses Sentence Transformers for semantic embeddings, FAISS for efficient similarity search, and Gradio for building an interactive chatbot interface.

These components enable Retrieval-Augmented Generation (RAG), allowing the chatbot to retrieve relevant hospitality information before generating responses.

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q gradio

# Compatible with sentence-transformers
!pip install -q "transformers>=4.41,<5"

!pip install -q peft
!pip install -q accelerate
!pip install -q bitsandbytes

!pip install -q evaluate
!pip install -q rouge-score
!pip install -q nltk

**Verify GPU Environment**

In [2]:
!nvidia-smi

Sat Jul  4 09:09:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print(torch.cuda.get_device_name(0))

Tesla T4


**Mount Google Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Required Libraries**

In [5]:
import os
import faiss
import torch
import numpy as np
import pandas as pd
import gradio as gr

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

In [6]:
import transformers
import sentence_transformers
import peft

print("Transformers          :", transformers.__version__)
print("Sentence Transformers :", sentence_transformers.__version__)
print("PEFT                  :", peft.__version__)

Transformers          : 4.57.6
Sentence Transformers : 5.6.0
PEFT                  : 0.19.1


**Configure Project Paths**

In [7]:
BASE_PATH = "/content/drive/MyDrive/Module_6_Specialization_NLP/HospitalityGPT"

DATA_PATH = f"{BASE_PATH}/data"

MODEL_PATH = f"{BASE_PATH}/models/hospitalitygpt"

RAG_PATH = f"{BASE_PATH}/rag"

INDEX_PATH = f"{RAG_PATH}/hospitality.index"

EMBEDDINGS_PATH = f"{RAG_PATH}/embeddings.npy"

DOCUMENTS_PATH = f"{RAG_PATH}/documents.csv"

os.makedirs(RAG_PATH, exist_ok=True)

**Load the Fine-Tuned HospitalityGPT Model**

The fine-tuned LoRA adapter is loaded on top of the original TinyLlama model.

Using LoRA adapters significantly reduces storage requirements while preserving the original model weights.

In [8]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [11]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [12]:
model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

### **Build the Hospitality Knowledge Base**

The Retrieval-Augmented Generation (RAG) system requires a searchable knowledge base.

Instead of storing only the customer question and response, each document also includes its business category and intent. This richer representation provides additional semantic context, helping the embedding model distinguish between similar customer queries belonging to different hospitality services.

The resulting documents form the foundation of the FAISS retrieval system.

In [13]:
knowledge_df = pd.read_csv(f"{DATA_PATH}/train_data.csv")

print(f"Knowledge Base Records: {len(knowledge_df):,}")

knowledge_df.head()

Knowledge Base Records: 20,184


,instruction,input,output,intent,category
0,id like to know more about the check-out could...,NaN,If you require further guidance on how to requ...,check_out,CHECK_OUT
1,"I need my fucking rom, how do I check it?",NaN,"To verify your hotel reservation, please adher...",check_hotel_reservation,HOTEL
2,would it be possible to speak with client serv...,NaN,To reach out to our customer service departmen...,customer_service,CONTACT
3,im looking for the fucking hotel offesr i need...,NaN,To provide you with the necessary guidance for...,check_hotel_offers,HOTEL
4,I would lik to know about the fucking check-in...,NaN,I am able to guide you through the process for...,check_in,CHECK_IN


Each knowledge document is constructed using:

- Business category
- Customer intent
- Customer question
- Expected response

This structure preserves domain-specific information while remaining readable and effective for semantic retrieval.

In [14]:
def build_document(row):

    return f"""
Category: {row['category']}

Intent: {row['intent']}

Customer Question:
{row['instruction']}

Answer:
{row['output']}
""".strip()


knowledge_df["document"] = knowledge_df.apply(
    build_document,
    axis=1
)

knowledge_df[["document"]].head()

,document
0,Category: CHECK_OUT\n\nIntent: check_out\n\nCu...
1,Category: HOTEL\n\nIntent: check_hotel_reserva...
2,Category: CONTACT\n\nIntent: customer_service\...
3,Category: HOTEL\n\nIntent: check_hotel_offers\...
4,Category: CHECK_IN\n\nIntent: check_in\n\nCust...


The knowledge documents are saved for future use.

Saving the processed knowledge base avoids rebuilding the document structure every time the notebook is executed and supports reproducibility.

In [15]:
knowledge_df.to_csv(DOCUMENTS_PATH, index=False)

print("Knowledge base saved successfully.")

Knowledge base saved successfully.


### **Generate Semantic Embeddings**

Sentence Transformers convert each knowledge document into a dense numerical representation called an embedding.

These embeddings capture semantic meaning rather than exact word matching, allowing the chatbot to retrieve relevant information even when users phrase questions differently.

In [16]:
# using one of the best lightweight embedding models available.

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

To improve efficiency, embeddings are generated only once.

If previously saved embeddings are detected, they are loaded directly from disk. Otherwise, new embeddings are computed and stored for future use.

In [17]:
if os.path.exists(EMBEDDINGS_PATH):

    print("Loading saved embeddings...")

    embeddings = np.load(EMBEDDINGS_PATH)

else:

    print("Generating embeddings...")

    embeddings = embedding_model.encode(
        knowledge_df["document"].tolist(),
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings Shape:", embeddings.shape)

Loading saved embeddings...
Embeddings Shape: (20184, 384)


### **Build the FAISS Index**

FAISS is used to perform efficient similarity search over the embedding vectors.

To reduce execution time during future runs, the index is saved to disk after it is created and automatically reloaded if it already exists.

In [18]:
dimension = embeddings.shape[1]

if os.path.exists(INDEX_PATH):

    print("Loading FAISS index...")

    index = faiss.read_index(INDEX_PATH)

else:

    print("Creating FAISS index...")

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)

print(f"Indexed Documents: {index.ntotal:,}")

Loading FAISS index...
Indexed Documents: 20,184


### **Test Semantic Retrieval**

Before integrating retrieval with the language model, the semantic search system is tested independently.

Given a user query, the embedding model retrieves the most relevant hospitality documents from the FAISS index.

In [19]:
def retrieve_context(query, top_k=4):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_docs = knowledge_df.iloc[
        indices[0]
    ]["document"].tolist()

    return retrieved_docs

In [20]:
query = "Can I request an early check-in?"

results = retrieve_context(query)

for i, doc in enumerate(results, 1):

    print("=" * 80)
    print(f"Retrieved Document {i}")
    print("=" * 80)
    print(doc)
    print()

Retrieved Document 1
Category: CHECK_IN

Intent: check_in

Customer Question:
would it be possible to ask for an early check-in?

Answer:
I am here to assist you with your inquiry regarding early check-in procedures. 1. Please visit our official website {{WEBSITE_URL}}. 2. Locate the {{MANAGE_BOOKING_SECTION}} on the homepage. 3. Provide your booking information to access your reservation. 4. Seek the {{REQUEST_EARLY_CHECK-IN_OPTION}} and complete the submission. 5. You will receive a confirmation email with the outcome of your request shortly. If you have any additional questions or require further support, please do not hesitate to contact us.

Retrieved Document 2
Category: CHECK_IN

Intent: check_in

Customer Question:
is it possible to ask for a fucking early check-in?

Answer:
I am here to provide guidance regarding your inquiry about early check-in. 1. Please visit our official website {{WEBSITE_URL}}. 2. Locate the {{MANAGE_BOOKING_SECTION}} on the homepage. 3. Input your reser

### **Construct the Retrieval-Augmented Prompt**

The retrieved hospitality documents are combined with the user's question to create a context-aware prompt.

Providing relevant context before generation enables the language model to produce more accurate and informative responses while reducing hallucinations.

Only the most relevant retrieved documents are included to keep the prompt concise and focused.

In [21]:
def build_rag_prompt(user_query, retrieved_docs):

    # Format retrieved documents nicely
    context = ""

    for i, doc in enumerate(retrieved_docs, 1):
        context += f"\nReference Document {i}\n"
        context += "-" * 50 + "\n"
        context += doc + "\n\n"

    prompt = f"""
You are HospitalityGPT, a professional AI assistant specializing in hotel and hospitality customer support.

Use ONLY the retrieved reference documents below to answer the customer's question.

Instructions:
- Answer naturally and conversationally.
- Be polite and professional.
- Provide a complete and informative response.
- Include step-by-step instructions whenever they are available.
- Use the retrieved documents only as background knowledge.
- DO NOT copy metadata such as Category, Intent, or Customer Question into your answer.
- If the answer is not available in the retrieved documents, politely state that you do not have enough information.

Retrieved Reference Documents:
{context}

Customer Question:
{user_query}

Detailed Answer:
"""

    return prompt

### **Generate Responses**

The retrieved context and customer question are passed to the fine-tuned HospitalityGPT model.

The model generates a response based on both its learned hospitality knowledge and the retrieved documents from the knowledge base.

This Retrieval-Augmented Generation (RAG) approach improves factual consistency compared to relying solely on the language model.

In [22]:
import re

def generate_response(user_query):

    # Retrieve relevant documents
    retrieved_docs = retrieve_context(user_query)

    # Build prompt
    prompt = build_rag_prompt(user_query, retrieved_docs)

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    # Generate response
    outputs = model.generate(

        **inputs,

        max_new_tokens=350,

        do_sample=True,

        temperature=0.4,

        top_p=0.9,

        repetition_penalty=1.15,

        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Extract only the answer
    if "Detailed Answer:" in response:
        answer = response.split("Detailed Answer:")[-1].strip()
    else:
        answer = response

    # Remove unwanted metadata if generated
    for marker in [
        "Category:",
        "Intent:",
        "Customer Question:",
        "Retrieved Reference Documents:",
        "Reference Document"
    ]:
        if marker in answer:
            answer = answer.split(marker)[0].strip()

    # Replace known placeholders
    replacements = {
        "{{WEBSITE_URL}}": "the hotel's official website",
        "{{INVOICES_SECTION}}": "the Invoices section of your account",
        "{{CHECK_OUT_OPTION}}": "Late Check-out option",
        "{{CHECK_IN_OPTION}}": "Early Check-in option",
        "{{RESERVATION_SECTION}}": "Reservations section",
        "{{BOOKING_REFERENCE}}": "your booking reference",
        "{{CUSTOMER_SUPPORT}}": "our customer support team",
        "{{SEARCH_BUTTON}}": "Search button",
        "{{LOGIN_BUTTON}}": "Log In button",
        "{{BOOK_NOW}}": "Book Now button",
        "{{CONFIRM_BUTTON}}": "Confirm button"
    }

    for placeholder, replacement in replacements.items():
        answer = answer.replace(placeholder, replacement)

    # Replace any remaining placeholders
    answer = re.sub(
        r"\{\{.*?\}\}",
        "the appropriate option",
        answer
    )

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Remove repeated punctuation
    answer = re.sub(r"\.{2,}", ".", answer)

    return answer.strip()

### **Test the RAG Pipeline**

Several hospitality-related customer queries are used to verify that the Retrieval-Augmented Generation pipeline retrieves relevant knowledge and generates appropriate responses.

These examples demonstrate the chatbot's ability to combine semantic retrieval with language generation.

In [23]:
test_questions = [

    "Can I check in early?",

    "How do I view my invoices?",

    "Is breakfast included?",

    "Can I cancel my reservation?",

    "Do you allow pets?",

    "Can I reserve airport transportation?"

]

for question in test_questions:

    print("="*100)
    print("Question:")
    print(question)

    docs = retrieve_context(question)

    print("\nRetrieved Context:")
    print("-"*100)

    for i, doc in enumerate(docs, 1):
        print(f"\nDocument {i}")
        print(doc)

    print("\nGenerated Response:")
    print("-"*100)

    print(generate_response(question))

    print("\n")

Question:
Can I check in early?

Retrieved Context:
----------------------------------------------------------------------------------------------------

Document 1
Category: FRONT_DESK

Intent: early_checkin

Customer Question:
Can I request an early check-in?

Answer:
Early check-in requests are accommodated based on room availability and cannot be guaranteed.

Document 2
Category: CHECK_IN

Intent: check_in

Customer Question:
is it possible to ask for a fucking early check-in?

Answer:
I am here to provide guidance regarding your inquiry about early check-in. 1. Please visit our official website {{WEBSITE_URL}}. 2. Locate the {{MANAGE_BOOKING_SECTION}} on the homepage. 3. Input your reservation information and choose your booking. 4. Find the {{REQUEST_EARLY_CHECK-IN_OPTION}} and complete your request. 5. Wait for a confirmation email that will inform you about the status of your request. If you require additional support, please feel free to contact us.

Document 3
Category: CHECK

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Certainly! Early check-in is available depending on the specific offerings of your hotel. 1. Access the official website of your chosen hotel. 2. Look for the the appropriate option on the homepage. 3. Confirm the dates over which you wish to make your early check-in request. 4. Proceed to submit your request according to the requirements provided. Should you have any more questions or require assistance, please do not hesitate to reach out.


Question:
How do I view my invoices?

Retrieved Context:
----------------------------------------------------------------------------------------------------

Document 1
Category: BILLING

Intent: invoices

Customer Question:
I'd like to see my invoices how cn i do it

Answer:
To view your invoices conveniently on our platform, please adhere to the following steps: 1. Go to {{WEBSITE_URL}} and log into your account by entering your username and password. 2. Locate the {{INVOICES_SECTION}} which houses all your billing documents. 3. Choose the par

### **Evaluate the Chatbot**

The chatbot is evaluated using a held-out test dataset.

Generated responses are compared with the reference responses using standard natural language generation metrics, including:

- BLEU Score
- ROUGE Score

These metrics provide an objective indication of response quality while complementing qualitative evaluation through manual testing.

In [24]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [25]:
test_df = pd.read_csv(f"{DATA_PATH}/test_data.csv")

# Evaluate on a representative sample of the test set
SAMPLE_SIZE = min(30, len(test_df))

evaluation_df = test_df.sample(
    SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

predictions = []
references = []

for i, (_, row) in enumerate(evaluation_df.iterrows(), start=1):

    print(f"Generating response {i}/{SAMPLE_SIZE}...", end="\r")

    prediction = generate_response(row["instruction"])

    predictions.append(prediction)
    references.append(row["output"])

print("\nEvaluation data generation complete.")

results_df = evaluation_df.copy()

results_df["prediction"] = predictions
results_df["reference"] = references

results_df.to_csv(
    f"{RAG_PATH}/evaluation_results.csv",
    index=False
)

print(f"\nSaved evaluation results to:\n{RAG_PATH}/evaluation_results.csv")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Evaluation data generation complete.

Saved evaluation results to:
/content/drive/MyDrive/Module_6_Specialization_NLP/HospitalityGPT/rag/evaluation_results.csv


In [26]:
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

print("="*60)
print("Evaluation Results")
print("="*60)

print(f"BLEU Score : {bleu_score['bleu']:.4f}")

print(f"ROUGE-1    : {rouge_score['rouge1']:.4f}")
print(f"ROUGE-2    : {rouge_score['rouge2']:.4f}")
print(f"ROUGE-L    : {rouge_score['rougeL']:.4f}")

Evaluation Results
BLEU Score : 0.2342
ROUGE-1    : 0.5330
ROUGE-2    : 0.2522
ROUGE-L    : 0.4285


The HospitalityGPT chatbot was evaluated using the BLEU and ROUGE metrics on a sample of the test dataset. BLEU measures the similarity between the generated response and the reference response based on word and phrase overlap, while ROUGE evaluates how well the generated response captures the important words and overall structure of the reference response.

The model achieved a **BLEU score of 0.2342**, **ROUGE-1 of 0.5330**, **ROUGE-2 of 0.2522**, and **ROUGE-L of 0.4285**.

Although the BLEU score appears relatively low, this is common for conversational AI systems because there are often multiple valid ways to answer the same question.

The ROUGE scores indicate that the chatbot is able to preserve important keywords and sentence structure while generating contextually relevant responses.

Overall, these results, together with qualitative testing, demonstrate that the fine-tuned RAG-based HospitalityGPT chatbot is capable of producing meaningful and domain-specific responses.


## **Deploy the HospitalityGPT Chatbot**

The final step is to deploy the Retrieval-Augmented Generation (RAG) pipeline as an interactive chatbot using Gradio.

The chatbot accepts natural language hospitality queries, retrieves relevant information from the knowledge base using FAISS, and generates context-aware responses using the fine-tuned HospitalityGPT model. This interface demonstrates the complete end-to-end functionality of the project.

In [27]:
def chatbot_response(message, history):

    response = generate_response(message)

    return response

In [28]:
examples = [
    ["Can I request an early check-in?"],
    ["How do I access my invoices?"],
    ["Is breakfast included with my booking?"],
    ["Can I cancel my reservation?"],
    ["Do you allow pets?"],
    ["Can I arrange airport transportation?"]
]

In [38]:
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🏨 HospitalityGPT",
    description=(
        "An AI-powered hospitality customer support chatbot built using "
        "TinyLlama, LoRA fine-tuning, FAISS, and Retrieval-Augmented Generation (RAG)."
    ),
    examples=examples,
    theme="soft"
)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [ ]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0bd735f23f5c4426d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## **Project Summary**

In this project, an intelligent HospitalityGPT chatbot was developed for the hospitality industry using a combination of Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG).

The project involved:

- Collecting and preprocessing a hospitality-specific instruction dataset.
- Fine-tuning the TinyLlama-1.1B Chat model using LoRA for parameter-efficient training.
- Building a semantic knowledge base using Sentence Transformers.
- Creating a FAISS vector database for efficient similarity search.
- Implementing a Retrieval-Augmented Generation pipeline to improve response accuracy.
- Evaluating the chatbot using BLEU and ROUGE metrics.
- Deploying the chatbot through an interactive Gradio interface.

The final chatbot demonstrates the ability to understand hospitality-related customer queries, retrieve relevant contextual information, and generate accurate, domain-specific responses, making it a practical AI assistant for hospitality customer support.